In [ ]:
"""
Advanced Inventory Analysis 
"""
import pandas as pd
import numpy as np
import hdbscan
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import adjusted_rand_score
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
INPUT_FILE = r"C:\Users\amir\OneDrive - TZANET INC\Bureau\AL-Inventory Mother.xlsm"

# ─────────────────────────────────────────
# READ DATE FROM Mother (L2)
# ─────────────────────────────────────────
summary_df = pd.read_excel(INPUT_FILE, sheet_name="Mother", header=None)
raw_value  = summary_df.iloc[1, 11]   # L2
dt         = pd.to_datetime(raw_value)
date_str   = dt.strftime("%Y_%m_%d-%H-%M-%S")
OUTPUT_FILE = rf"C:\Users\amir\OneDrive - TZANET INC\Bureau\AL-Tzanet_Analysis_{date_str}.xlsx"

SHEET_NAME       = "Mother"
HEADER_ROW       = 2
LEAD_TIME_DAYS   = 30
HOLDING_COST_PCT = 0.20
ORDER_COST       = 50
DAYS_OBSOLETE    = 365
DAYS_DEAD        = 730
DISCOUNT_RATE    = 0.10

def get_liquidation_disc(days_since_sale, item_group=''):
    base = 0.20
    if days_since_sale > DAYS_DEAD:
        base = 0.50
    elif days_since_sale > DAYS_OBSOLETE:
        base = 0.35
    grp = str(item_group).lower()
    if any(x in grp for x in ['electronic', 'tech', 'digit']):
        base = min(base + 0.10, 0.70)
    elif any(x in grp for x in ['part', 'spare', 'piece']):
        base = max(base - 0.05, 0.10)
    return base

# ─────────────────────────────────────────
# 1. LOAD & CLEAN
# ─────────────────────────────────────────
print("Loading data...")
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=HEADER_ROW)
df = df.dropna(how='all')
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df.columns = df.columns.str.strip()
date_cols = df.select_dtypes(include=['datetime64']).columns
df = df.drop(columns=date_cols, errors='ignore')

sold_cols   = ['QtySold_0_12M','QtySold_12_24M','QtySold_24_36M','QtySold_36_48M','QtySold_48_60M']
purch_cols  = ['QtyPurchased_0_12M','QtyPurchased_12_24M','QtyPurchased_24_36M','QtyPurchased_36_48M','QtyPurchased_48_60M']
nsale_cols  = ['NetSalesValue_0_12M','NetSalesValue_12_24M','NetSalesValue_24_36M','NetSalesValue_36_48M','NetSalesValue_48_60M']
npurch_cols = ['NetPurchaseValue_0_12M','NetPurchaseValue_12_24M','NetPurchaseValue_24_36M','NetPurchaseValue_36_48M','NetPurchaseValue_48_60M']

for c in sold_cols + purch_cols + nsale_cols + npurch_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

for c in ['OnHand','ItemCost','RetailPrice','VendorPrice','LastSalePrice',
          'LastPurchasePrice','DaysSinceLastSale','DaysSinceLastPurchase',
          'MinStock','IsCommited','OnOrder','Available']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

# ─────────────────────────────────────────
# 2. BASE METRICS
# ─────────────────────────────────────────
df['TotalSold_5Y']          = df[[c for c in sold_cols   if c in df.columns]].sum(axis=1)
df['TotalPurchased_5Y']     = df[[c for c in purch_cols  if c in df.columns]].sum(axis=1)
df['TotalSalesValue_5Y']    = df[[c for c in nsale_cols  if c in df.columns]].sum(axis=1)
df['TotalPurchaseValue_5Y'] = df[[c for c in npurch_cols if c in df.columns]].sum(axis=1)

avg_real = df['TotalPurchaseValue_5Y'] / df['TotalPurchased_5Y'].replace(0, np.nan)
df['AvgPurchasePrice'] = np.where(
    df['ItemCost'].fillna(0) > 0,
    df['ItemCost'],
    np.where(
        (avg_real > 0) & (avg_real < 1e6),
        avg_real,
        df['ItemCost'].fillna(0)
    )
)

df['Profit']         = df['RetailPrice'].fillna(0) - df['ItemCost'].fillna(0)
df['MarginPct']      = df['Profit'] / df['RetailPrice'].replace(0, np.nan) * 100
df['InventoryValue'] = df['InventoryValue'].fillna(0)
df['MonthlySales']   = df['TotalSold_5Y'] / 60

# ─────────────────────────────────────────
# 3. ABC ANALYSIS
# ─────────────────────────────────────────
KEY_COLS = ['ItemCode', 'WhsCode'] if 'WhsCode' in df.columns else ['ItemCode']
df_abc = df[KEY_COLS + ['TotalSalesValue_5Y']].copy()
df_abc = df_abc.sort_values('TotalSalesValue_5Y', ascending=False)
df_abc['CumSalesPct'] = (
    df_abc['TotalSalesValue_5Y'].cumsum()
    / df_abc['TotalSalesValue_5Y'].sum() * 100
)
df_abc['ABC_Class'] = np.where(
    df_abc['CumSalesPct'] <= 80, 'A',
    np.where(df_abc['CumSalesPct'] <= 95, 'B', 'C')
)
df = df.merge(df_abc[KEY_COLS + ['ABC_Class']], on=KEY_COLS, how='left')
df['ABC_Class'] = df['ABC_Class'].fillna('C')

if 'WhsCode' in df.columns:
    item_sales = df.groupby('ItemCode')['TotalSalesValue_5Y'].sum().reset_index()
    item_sales = item_sales.sort_values('TotalSalesValue_5Y', ascending=False)
    item_sales['CumPct'] = item_sales['TotalSalesValue_5Y'].cumsum() / item_sales['TotalSalesValue_5Y'].sum() * 100
    item_sales['ABC_Item_Level'] = np.where(
        item_sales['CumPct'] <= 80, 'A',
        np.where(item_sales['CumPct'] <= 95, 'B', 'C')
    )
    df = df.merge(item_sales[['ItemCode','ABC_Item_Level']], on='ItemCode', how='left')
    df['ABC_Item_Level'] = df['ABC_Item_Level'].fillna('C')

# ─────────────────────────────────────────
# 4. XYZ ANALYSIS
# ─────────────────────────────────────────
avail_sold = [c for c in sold_cols if c in df.columns]
sold_arr   = df[avail_sold].values
sold_mean  = sold_arr.mean(axis=1)
sold_std   = sold_arr.std(axis=1)
df['SalesCV']   = np.where(sold_mean > 0, sold_std / sold_mean, 99)
df['XYZ_Class'] = np.where(df['SalesCV'] < 0.5, 'X',
                   np.where(df['SalesCV'] < 1.0, 'Y', 'Z'))
df['ABC_XYZ']   = df['ABC_Class'] + df['XYZ_Class']

# ─────────────────────────────────────────
# 5. SALES VELOCITY
# ─────────────────────────────────────────
df['VelocityRatio'] = (
    df['QtySold_0_12M'] / df['QtySold_12_24M'].replace(0, np.nan)
).fillna(0) if 'QtySold_0_12M' in df.columns and 'QtySold_12_24M' in df.columns else 0

df['Velocity_Trend'] = np.where(
    df['VelocityRatio'] >= 1.2, 'Growing',
    np.where(df['VelocityRatio'] >= 0.8, 'Stable',
    np.where(df['VelocityRatio'] > 0,    'Declining', 'No_Recent_Sales'))
)

periods_vec = np.array([1, 2, 3, 4, 5], dtype=float)
if len(avail_sold) == 5:
    Y = df[list(reversed(avail_sold))].values.T
    X_design = np.vstack([periods_vec, np.ones(5)]).T
    coefs, _, _, _ = np.linalg.lstsq(X_design, Y, rcond=None)
    df['Sales_Slope_5Y'] = coefs[0]
else:
    df['Sales_Slope_5Y'] = 0.0

# ─────────────────────────────────────────
# 6. OBSOLESCENCE RISK
# ─────────────────────────────────────────
df['ObsolescenceScore'] = 0.0
if 'DaysSinceLastSale' in df.columns:
    df['ObsolescenceScore'] += np.where(
        df['DaysSinceLastSale'] > DAYS_DEAD,     40,
        np.where(df['DaysSinceLastSale'] > DAYS_OBSOLETE, 20, 0)
    )
df['ObsolescenceScore'] += np.where(df['TotalSold_5Y'] == 0, 30, 0)
df['ObsolescenceScore'] += np.where(df['VelocityRatio'] == 0, 20, 0)
if 'Status' in df.columns:
    df['ObsolescenceScore'] += np.where(
        (df['Status'] == 'Inactive') & (df['OnHand'] != 0), 10, 0
    )
df['Obsolescence_Risk'] = np.where(
    df['ObsolescenceScore'] >= 70, 'Critical',
    np.where(df['ObsolescenceScore'] >= 40, 'High',
    np.where(df['ObsolescenceScore'] >= 20, 'Medium', 'Low'))
)

# ─────────────────────────────────────────
# 7. REORDER POINT & SAFETY STOCK
# ─────────────────────────────────────────
daily_sales      = df['TotalSold_5Y'] / (60 * 30)
demand_std_daily = sold_std / (60 * 30)
z_score          = 1.65
df['SafetyStock']  = (z_score * demand_std_daily * np.sqrt(LEAD_TIME_DAYS)).round(0)
df['ReorderPoint'] = (daily_sales * LEAD_TIME_DAYS + df['SafetyStock']).round(0)
df['Reorder_Alert'] = np.where(
    df['OnHand'] <= df['ReorderPoint'],           'REORDER NOW',
    np.where(df['OnHand'] <= df['ReorderPoint'] * 1.2, 'Watch', 'OK')
)

# ─────────────────────────────────────────
# 8. EOQ
# ─────────────────────────────────────────
annual_demand     = df['TotalSold_5Y'] / 5
holding_cost_unit = df['ItemCost'].fillna(0) * HOLDING_COST_PCT
eoq_denom         = holding_cost_unit.replace(0, np.nan)
df['EOQ'] = np.where(
    annual_demand > 0,
    np.sqrt((2 * annual_demand * ORDER_COST) / eoq_denom.fillna(1)),
    0
).round(0)
df['EOQ'] = df['EOQ'].fillna(0)

# ─────────────────────────────────────────
# 9. INVENTORY TURNOVER & DSI
# ─────────────────────────────────────────
df['Turnover_Rate_Raw'] = (
    df['TotalSalesValue_5Y'] / df['InventoryValue'].replace(0, np.nan) / 5
).fillna(0)
df['Turnover_Rate'] = df['Turnover_Rate_Raw'].clip(lower=0, upper=365)
df['Data_Quality_Flag'] = np.where(
    df['Turnover_Rate_Raw'] < 0,            'Negative Inventory (SAP Error)',
    np.where(df['Turnover_Rate_Raw'] > 365, 'Abnormal Turnover (Data Error)',
    np.where(df['InventoryValue']    < 0,   'Negative Inventory Value',
    np.where((df['MarginPct'] >= 90) | (df['MarginPct'] <= -90), 'Abnormal Margin',
    'OK')))
)
df['DSI'] = np.where(df['Turnover_Rate'] > 0, 365 / df['Turnover_Rate'], 9999)
df['Turnover_Category'] = np.where(
    df['Turnover_Rate'] >= 6, 'Fast Mover',
    np.where(df['Turnover_Rate'] >= 2, 'Normal',
    np.where(df['Turnover_Rate'] > 0,  'Slow Mover', 'No Movement'))
)

# ─────────────────────────────────────────
# 10. CASH FLOW & WRITE-OFF
# ─────────────────────────────────────────
last_sale_price = df['LastSalePrice'].fillna(df['RetailPrice']) if 'LastSalePrice' in df.columns else df['RetailPrice']
df['InvestedCapital'] = df['OnHand'] * df['ItemCost'].fillna(0)
days_col = df['DaysSinceLastSale'] if 'DaysSinceLastSale' in df.columns else pd.Series(0, index=df.index)
grp_col  = df['ItemGroupName']     if 'ItemGroupName'     in df.columns else pd.Series('', index=df.index)
df['LiquidationDisc'] = [
    get_liquidation_disc(d, g)
    for d, g in zip(days_col, grp_col)
]
df['LiquidationValue']    = df['OnHand'] * last_sale_price * (1 - df['LiquidationDisc'])
df['CashRelease_If_Sold'] = df['LiquidationValue']
df['WriteOff_Risk_Value'] = np.where(
    df['Obsolescence_Risk'].isin(['Critical','High']), df['InvestedCapital'], 0
)
df['WriteOff_Candidate'] = (
    (df['Obsolescence_Risk'] == 'Critical') |
    ((df['Obsolescence_Risk'] == 'High') & (df['TotalSold_5Y'] == 0))
)

# ─────────────────────────────────────────
# 11. MARGIN SQUEEZE
# ─────────────────────────────────────────
if 'LastSalePrice' in df.columns:
    df['PriceRealizationPct'] = (
        df['LastSalePrice'] / df['RetailPrice'].replace(0, np.nan) * 100
    ).fillna(100)
    df['Margin_Squeeze'] = np.where(
        df['PriceRealizationPct'] < 80,  'Severe (< 80%)',
        np.where(df['PriceRealizationPct'] < 90,  'Moderate (80-90%)',
        np.where(df['PriceRealizationPct'] < 100, 'Slight (90-100%)', 'OK'))
    )
    df['DiscountLoss'] = (
        df['RetailPrice'] - df['LastSalePrice'].fillna(df['RetailPrice'])
    ) * df['TotalSold_5Y']

# ─────────────────────────────────────────
# 12. SUPPLIER ANALYSIS
# ─────────────────────────────────────────
if 'LastPurchasePrice' in df.columns:
    df['PriceDrift_Pct'] = (
        (df['LastPurchasePrice'] - df['AvgPurchasePrice'])
        / df['AvgPurchasePrice'].replace(0, np.nan) * 100
    ).fillna(0)
    df['Supplier_Price_Trend'] = np.where(
        df['PriceDrift_Pct'] > 10,   'Increasing (>10%)',
        np.where(df['PriceDrift_Pct'] > 0,    'Slightly Up',
        np.where(df['PriceDrift_Pct'] < -10,  'Decreasing (>10%)', 'Stable'))
    )

if 'CardCode' in df.columns:
    sup_agg = df.groupby('CardCode').agg(
        SKU_Count          = ('ItemCode',              'nunique'),
        TotalPurchased     = ('TotalPurchased_5Y',     'sum'),
        TotalPurchValue    = ('TotalPurchaseValue_5Y', 'sum'),
        AvgPriceDrift      = ('PriceDrift_Pct',        'mean') if 'PriceDrift_Pct' in df.columns else ('ItemCode', 'count'),
        ObsRisk_High       = ('Obsolescence_Risk',     lambda x: (x.isin(['Critical','High'])).sum()),
        DaysSinceLastPurch = ('DaysSinceLastPurchase', 'mean') if 'DaysSinceLastPurchase' in df.columns else ('ItemCode', 'count'),
    ).reset_index()
    if 'AvgPriceDrift' in sup_agg.columns:
        sup_agg['Price_Score'] = np.clip(100 - sup_agg['AvgPriceDrift'] * 2, 0, 100)
    else:
        sup_agg['Price_Score'] = 50
    sup_agg['Obs_Score']    = np.clip(100 - sup_agg['ObsRisk_High'] / sup_agg['SKU_Count'].replace(0,1) * 100, 0, 100)
    sup_agg['Volume_Score'] = np.clip(sup_agg['TotalPurchased'] / sup_agg['TotalPurchased'].max() * 100, 0, 100)
    sup_agg['Supplier_Score'] = (
        sup_agg['Price_Score']  * 0.40 +
        sup_agg['Obs_Score']    * 0.40 +
        sup_agg['Volume_Score'] * 0.20
    ).round(1)
    sup_agg['Supplier_Tier'] = np.where(
        sup_agg['Supplier_Score'] >= 75, 'Tier 1 — Strategic',
        np.where(sup_agg['Supplier_Score'] >= 50, 'Tier 2 — Standard', 'Tier 3 — Review')
    )

# ─────────────────────────────────────────
# 13. WEB vs OFFLINE
# ─────────────────────────────────────────
if 'WebActive' in df.columns:
    df['WebActive'] = (
        df['WebActive'].astype(str).str.strip().str.lower()
        .replace(['nan','none',''], '0')
        .map({'yes':1,'y':1,'true':1,'1':1,'no':0,'n':0,'false':0,'0':0})
        .fillna(0).astype(int)
    )

# ─────────────────────────────────────────
# 14. SEASONALITY — DETRENDED
# ─────────────────────────────────────────
period_label = {
    'QtySold_0_12M':  'Last 12M',
    'QtySold_12_24M': '12-24M ago',
    'QtySold_24_36M': '24-36M ago',
    'QtySold_36_48M': '36-48M ago',
    'QtySold_48_60M': '48-60M ago',
}
if len(avail_sold) >= 3:
    sales_arr    = df[list(reversed(avail_sold))].values.astype(float)
    t            = np.arange(sales_arr.shape[1], dtype=float)
    t_mean       = t.mean()
    y_mean       = sales_arr.mean(axis=1, keepdims=True)
    slope_s      = ((sales_arr - y_mean) * (t - t_mean)).sum(axis=1) / ((t - t_mean)**2).sum()
    intercept_s  = y_mean.flatten() - slope_s * t_mean
    trend_component = np.outer(slope_s, t) + intercept_s[:, None]
    detrended    = sales_arr - trend_component
    residual_std = detrended.std(axis=1)
    sales_avg    = sales_arr.mean(axis=1) + 1e-6
    df['SeasonalityIndex']     = residual_std / sales_avg
    df['Is_Seasonal']          = df['SeasonalityIndex'] > 0.3
    df['Seasonality_Strength'] = np.where(
        df['SeasonalityIndex'] > 0.8, 'Strong',
        np.where(df['SeasonalityIndex'] > 0.3, 'Moderate', 'Low')
    )
else:
    df['SeasonalityIndex']     = 0.0
    df['Is_Seasonal']          = False
    df['Seasonality_Strength'] = 'Low'

peak_idx          = df[avail_sold].idxmax(axis=1)
df['Peak_Period'] = peak_idx.map(period_label).fillna('N/A')

# ─────────────────────────────────────────
# NEW A: WEIGHTED MOVING AVERAGE FORECAST
# ─────────────────────────────────────────
if len(avail_sold) >= 2:
    weights   = np.array([5, 4, 3, 2, 1], dtype=float)[:len(avail_sold)]
    weights   = weights / weights.sum()
    sales_wma = df[list(reversed(avail_sold))].values
    df['WMA_Forecast_12M']      = (sales_wma * weights).sum(axis=1).round(0)
    df['TrendAdj_Forecast_12M'] = np.maximum(
        (df['WMA_Forecast_12M'] + df['Sales_Slope_5Y']).round(0), 0
    )
    if 'QtySold_0_12M' in df.columns:
        df['Forecast_vs_Actual_Pct'] = (
            (df['TrendAdj_Forecast_12M'] - df['QtySold_0_12M'])
            / df['QtySold_0_12M'].replace(0, np.nan) * 100
        ).fillna(0).round(1)
        df['Forecast_Signal'] = np.where(
            df['Forecast_vs_Actual_Pct'] > 20,   'Demand Growing',
            np.where(df['Forecast_vs_Actual_Pct'] < -20, 'Demand Shrinking', 'Stable Demand')
        )

# ─────────────────────────────────────────
# NEW C: MULTI-ECHELON INVENTORY
# ─────────────────────────────────────────
if 'WhsCode' in df.columns:
    item_total_sold = df.groupby('ItemCode')['TotalSold_5Y'].transform('sum')
    df['Whs_DemandShare_Pct'] = np.where(
        item_total_sold > 0,
        df['TotalSold_5Y'] / item_total_sold * 100, 0
    ).round(1)
    item_total_inv = df.groupby('ItemCode')['OnHand'].transform('sum')
    df['Optimal_OnHand_Whs']  = (item_total_inv * df['Whs_DemandShare_Pct'] / 100).round(0)
    df['Inventory_Imbalance'] = df['OnHand'] - df['Optimal_OnHand_Whs']
    df['Rebalance_Action']    = np.where(
        df['Inventory_Imbalance'] > 10,    'Transfer Out',
        np.where(df['Inventory_Imbalance'] < -10, 'Transfer In', 'Balanced')
    )

# ─────────────────────────────────────────
# NEW D: DEAD STOCK NPV RECOVERY
# ─────────────────────────────────────────
monthly_disc = (1 + DISCOUNT_RATE) ** (1/12) - 1
df['NPV_Liquidate_Now']       = df['CashRelease_If_Sold']
df['NPV_Liquidate_3M']        = df['CashRelease_If_Sold'] * 0.95 / (1 + monthly_disc) ** 3
df['NPV_Liquidate_6M']        = df['CashRelease_If_Sold'] * 0.90 / (1 + monthly_disc) ** 6
df['NPV_Liquidate_12M']       = df['CashRelease_If_Sold'] * 0.80 / (1 + monthly_disc) ** 12
df['NPV_Cost_of_Waiting_6M']  = (df['NPV_Liquidate_Now'] - df['NPV_Liquidate_6M']).clip(lower=0)
df['NPV_Cost_of_Waiting_12M'] = (df['NPV_Liquidate_Now'] - df['NPV_Liquidate_12M']).clip(lower=0)
df['Best_Liquidation_Window'] = np.where(
    df['Obsolescence_Risk'] == 'Critical', 'Immediately',
    np.where(df['NPV_Cost_of_Waiting_6M']  > df['InvestedCapital'] * 0.05, 'Within 3 Months',
    np.where(df['NPV_Cost_of_Waiting_12M'] > df['InvestedCapital'] * 0.05, 'Within 6 Months', 'Can Wait'))
)

# ─────────────────────────────────────────
# NEW G: COST VARIANCE ACROSS WAREHOUSES
# ─────────────────────────────────────────
cost_variance_df = pd.DataFrame()
cost_detail_df   = pd.DataFrame()
if 'WhsCode' in df.columns and 'ItemCost' in df.columns:
    print("Calculating Cost Variance across warehouses...")
    df_cost = df[df['ItemCost'] > 0][['ItemCode','ItemName','WhsCode','ItemCost','OnHand','InventoryValue']].copy()
    cost_stats = df_cost.groupby('ItemCode').agg(
        ItemName      = ('ItemName',       'first'),
        Whs_Count     = ('WhsCode',        'nunique'),
        Cost_Min      = ('ItemCost',       'min'),
        Cost_Max      = ('ItemCost',       'max'),
        Cost_Mean     = ('ItemCost',       'mean'),
        Cost_Std      = ('ItemCost',       'std'),
        TotalOnHand   = ('OnHand',         'sum'),
        TotalInvValue = ('InventoryValue', 'sum'),
    ).reset_index()
    cost_stats = cost_stats[cost_stats['Whs_Count'] > 1].copy()
    cost_stats['Cost_Variance_Pct'] = (
        (cost_stats['Cost_Max'] - cost_stats['Cost_Min'])
        / cost_stats['Cost_Min'].replace(0, np.nan) * 100
    ).fillna(0).round(1)
    cost_stats['Cost_Gap_Abs']    = (cost_stats['Cost_Max'] - cost_stats['Cost_Min']).round(2)
    cost_stats['Financial_Impact'] = (cost_stats['Cost_Gap_Abs'] * cost_stats['TotalOnHand']).round(0)
    cost_stats['Variance_Severity'] = np.where(
        cost_stats['Cost_Variance_Pct'] >= 50, ' Critical (≥50%)',
        np.where(cost_stats['Cost_Variance_Pct'] >= 20, ' High (20-50%)',
        np.where(cost_stats['Cost_Variance_Pct'] >= 10, ' Medium (10-20%)',
        ' Low (<10%)'))
    )
    cost_variance_df = cost_stats[cost_stats['Cost_Variance_Pct'] >= 5].sort_values(
        'Financial_Impact', ascending=False
    ).reset_index(drop=True)
    top_items      = cost_variance_df.head(200)['ItemCode'].tolist()
    cost_detail_df = df_cost[df_cost['ItemCode'].isin(top_items)].copy()
    cost_detail_df = cost_detail_df.sort_values(['ItemCode','ItemCost'], ascending=[True,False])
    print(f"    Found {len(cost_variance_df):,} items with cost variance ≥5% across warehouses")

# ─────────────────────────────────────────
# 15. ML SEGMENTATION — HDBSCAN
# ─────────────────────────────────────────
print("Running ML Segmentation (HDBSCAN)...")

# ── Feature Engineering ───────────────────────────────────────────────────────
df['DaysOfInventory'] = (df['OnHand'] / df['MonthlySales'].replace(0, np.nan)) * 30
df['GMROI']           = df['TotalSalesValue_5Y'] / df['InventoryValue'].replace(0, np.nan)
sold_cols_local       = [c for c in df.columns if 'QtySold_' in c]
df['SalesFrequency']  = (df[sold_cols_local] > 0).sum(axis=1) / max(len(sold_cols_local), 1)
df['DemandVariability'] = df.get('SalesCV', 0)
df['RecencyScore']    = 1 / (df.get('DaysSinceLastSale', pd.Series(0, index=df.index)) + 1)
df['ProfitValue']     = df['TotalSalesValue_5Y'] * (df['MarginPct'] / 100)

df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

features = [
    'TotalSold_5Y', 'InventoryValue', 'MarginPct', 'Turnover_Rate',
    'ObsolescenceScore', 'VelocityRatio', 'DaysOfInventory', 'GMROI',
    'SalesFrequency', 'DemandVariability', 'RecencyScore', 'ProfitValue'
]
features = [f for f in features if f in df.columns]

X = df[features].copy()
for col in ['TotalSold_5Y', 'InventoryValue', 'ProfitValue']:
    if col in X.columns:
        X[col] = np.log1p(X[col].clip(lower=0))

scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X)

# ── HDBSCAN ──────────────────────────────────────────────────────────────────
min_cls  = max(30, int(len(df) * 0.01))
clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cls, min_samples=10)
df['ML_Cluster'] = clusterer.fit_predict(X_scaled)

n_clusters = len(set(df['ML_Cluster'])) - (1 if -1 in df['ML_Cluster'].values else 0)
noise_cnt  = (df['ML_Cluster'] == -1).sum()
print(f"    Clusters found : {n_clusters}")
print(f"     Noise points  : {noise_cnt:,}")

# ── Cluster Profile ───────────────────────────────────────────────────────────
cluster_profile = df.groupby('ML_Cluster')[features].median().reset_index()

def categorize(series):
    try:
        return pd.qcut(series, q=3, labels=['Low', 'Mid', 'High'])
    except Exception:
        return pd.Series(['Mid'] * len(series), index=series.index)

for col in features:
    cluster_profile[col + '_Level'] = categorize(cluster_profile[col])

def build_label(row):
    parts = []
    if row.get('TotalSold_5Y_Level') == 'High':
        parts.append('High Sales')
    elif row.get('TotalSold_5Y_Level') == 'Low':
        parts.append('Low Sales')
    if row.get('MarginPct_Level') == 'High':
        parts.append('High Margin')
    if row.get('Turnover_Rate_Level') == 'High':
        parts.append('Fast Moving')
    if row.get('ObsolescenceScore_Level') == 'High':
        parts.append('At Risk')
    if row.get('DaysOfInventory_Level') == 'High':
        parts.append('Overstock')
    if row.get('GMROI_Level') == 'High':
        parts.append('Efficient')
    return ' | '.join(parts) if parts else 'Balanced'

cluster_profile['ML_Label'] = cluster_profile.apply(build_label, axis=1)

# ── Resolve duplicate labels ─────────────────────────────────────────────────
from collections import Counter

def resolve_duplicates_hdbscan(cluster_profile):
    label_counts     = Counter(cluster_profile['ML_Label'])
    duplicate_labels = {l for l, c in label_counts.items() if c > 1}
    if not duplicate_labels:
        return cluster_profile

    result = cluster_profile.copy()
    sort_priority = {
        'High Sales':   ('TotalSold_5Y',      False),
        'Low Sales':    ('TotalSold_5Y',      True),
        'High Margin':  ('MarginPct',         False),
        'Fast Moving':  ('Turnover_Rate',     False),
        'At Risk':      ('ObsolescenceScore', False),
        'Overstock':    ('DaysOfInventory',   False),
        'Efficient':    ('GMROI',             False),
        'Balanced':     ('TotalSold_5Y',      False),
    }
    tier_suffixes = ['— Tier 1', '— Tier 2', '— Tier 3', '— Tier 4', '— Tier 5']

    for dup_label in duplicate_labels:
        mask = result['ML_Label'] == dup_label
        rows = result[mask].copy()
        first_word = dup_label.split(' | ')[0] if ' | ' in dup_label else dup_label
        sort_col, asc = sort_priority.get(first_word, ('TotalSold_5Y', False))
        if sort_col in rows.columns:
            rows = rows.sort_values(sort_col, ascending=asc)
        for i, idx in enumerate(rows.index):
            suffix = tier_suffixes[i] if i < len(tier_suffixes) else f'— Tier {i+1}'
            result.loc[idx, 'ML_Label'] = f"{dup_label} {suffix}"

    return result

cluster_profile = resolve_duplicates_hdbscan(cluster_profile)

label_map       = dict(zip(cluster_profile['ML_Cluster'], cluster_profile['ML_Label']))
df['ML_Segment'] = df['ML_Cluster'].map(label_map).fillna('Noise — Unclustered')

# ── Feature Importance ───────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_scaled, df['ML_Cluster'])
importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("\n Feature Importance:")
print(importance.round(3))

# ── Stability Check ──────────────────────────────────────────────────────────
try:
    n_sub      = int(len(X_scaled) * 0.8)
    sub_idx    = np.random.choice(len(X_scaled), n_sub, replace=False)
    labels_sub = hdbscan.HDBSCAN(min_cluster_size=min_cls, min_samples=10).fit_predict(X_scaled[sub_idx])
    labels_ref = df['ML_Cluster'].values[sub_idx]
    ari_score  = adjusted_rand_score(labels_ref, labels_sub)
    print(f"\n Stability (ARI on 80% subsample): {ari_score:.4f}")
except Exception:
    print("\n Stability check skipped")

# ── ML Summary ───────────────────────────────────────────────────────────────
ml_seg_summary = df.groupby('ML_Segment').agg(
    Count      = ('ML_Cluster',   'count'),
    TotalSales = ('TotalSold_5Y', 'sum'),
    AvgMargin  = ('MarginPct',    'mean'),
    AvgTurnover= ('Turnover_Rate','mean'),
).sort_values('TotalSales', ascending=False)
print("\n ML Segment Summary:")
print(ml_seg_summary)

# ─────────────────────────────────────────
# 16. ANOMALY DETECTION
# ─────────────────────────────────────────
anomaly_features = ['TotalSold_5Y','VelocityRatio','MarginPct','Turnover_Rate']
anomaly_features = [f for f in anomaly_features if f in df.columns]
X_anomaly = df[anomaly_features].replace([np.inf,-np.inf], 0).fillna(0)
iso = IsolationForest(contamination=0.05, random_state=42)
df['Anomaly_Flag']  = iso.fit_predict(X_anomaly)
df['Is_Anomaly']    = df['Anomaly_Flag'] == -1
df['Anomaly_Score'] = iso.score_samples(X_anomaly)

# ─────────────────────────────────────────
# 17. COMPREHENSIVE ACTION
# ─────────────────────────────────────────
def get_action(row):
    if row.get('WriteOff_Candidate', False):
        return 'WRITE-OFF CANDIDATE'
    if row.get('Reorder_Alert','') == 'REORDER NOW':
        return 'URGENT: Reorder Now'
    if row.get('Obsolescence_Risk','') == 'Critical':
        return 'Liquidate Immediately'
    if row.get('ABC_Class','') == 'A' and row.get('XYZ_Class','') == 'X':
        return 'Top Priority: Maintain Stock'
    if row.get('Velocity_Trend','') == 'Growing' and row.get('ABC_Class','') in ['A','B']:
        return 'Invest: Growing Demand'
    if row.get('Velocity_Trend','') == 'Declining' and row.get('ABC_Class','') == 'C':
        return 'Phase Out'
    if str(row.get('Margin_Squeeze','')).startswith('Severe'):
        return 'Review Pricing / Costs'
    if row.get('Turnover_Category','') == 'No Movement':
        return 'Investigate: Zero Movement'
    if row.get('Rebalance_Action','') == 'Transfer Out':
        return 'Rebalance: Transfer Excess to Another Whs'
    if row.get('Reorder_Alert','') == 'Watch':
        return 'Monitor Stock Level'
    return 'Normal Operations'

df['Advanced_Action'] = df.apply(get_action, axis=1)

# ─────────────────────────────────────────
# 18. HELPER FUNCTIONS
# ─────────────────────────────────────────
def fmt_header(ws, color='1F4E79'):
    fill   = PatternFill('solid', start_color=color)
    font   = Font(bold=True, color='FFFFFF', name='Arial', size=10)
    align  = Alignment(horizontal='center', vertical='center', wrap_text=True)
    thin   = Side(style='thin', color='CCCCCC')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for cell in ws[1]:
        cell.fill = fill; cell.font = font
        cell.alignment = align; cell.border = border
    ws.row_dimensions[1].height = 30

def auto_width(ws, max_w=40, col_overrides=None):
    col_overrides = col_overrides or {}
    for col in ws.columns:
        header = str(col[0].value) if col[0].value else ''
        if header in col_overrides:
            ws.column_dimensions[col[0].column_letter].width = col_overrides[header]
        else:
            max_len = max((len(str(c.value)) for c in col if c.value), default=8)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, max_w)

# ─────────────────────────────────────────
# 19. EXCEL OUTPUT
# ─────────────────────────────────────────
print("Writing Excel file...")
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    wb = writer.book

    exec_data = {
        'Metric': [
            'Total Inventory Value',
            'Total Value at Risk of Write-off',
            'Total Cash Recoverable from DeadStock',
            'NPV Cost of Waiting 6 Months (Dead Stock)',
            'Total # of SKUs/Whs Records',
            '# SKUs/Whs with No Sales in 12 Months',
            '# SKUs/Whs Requiring Immediate Reorder',
            '# SKUs/Whs with Severe Margin Squeeze',
            '# Anomalies Detected',
            '# Seasonal Items',
            '# Items with Demand Growing (Forecast)',
            '# Items with Data Quality Issues',
            '# Items Needing Rebalancing',
            '# Items with Cost Variance ≥5% Across Whs',
            'HDBSCAN Clusters Found',
            'HDBSCAN Noise Points',
        ],
        'Value': [
            f"${df['InventoryValue'].sum():,.0f}",
            f"${df['WriteOff_Risk_Value'].sum():,.0f}",
            f"${df[df['Obsolescence_Risk'].isin(['Critical','High'])]['CashRelease_If_Sold'].sum():,.0f}",
            f"${df[df['Obsolescence_Risk'].isin(['Critical','High'])]['NPV_Cost_of_Waiting_6M'].sum():,.0f}",
            f"{len(df):,}",
            f"{(df['QtySold_0_12M'] == 0).sum():,}" if 'QtySold_0_12M' in df.columns else 'N/A',
            f"{(df['Reorder_Alert'] == 'REORDER NOW').sum():,}",
            f"{df['Margin_Squeeze'].str.startswith('Severe').sum():,}" if 'Margin_Squeeze' in df.columns else 'N/A',
            f"{df['Is_Anomaly'].sum():,}",
            f"{df['Is_Seasonal'].sum():,}",
            f"{(df['Forecast_Signal'] == 'Demand Growing').sum():,}" if 'Forecast_Signal' in df.columns else 'N/A',
            f"{(df['Data_Quality_Flag'] != 'OK').sum():,}",
            f"{(df['Rebalance_Action'] != 'Balanced').sum():,}" if 'Rebalance_Action' in df.columns else 'N/A',
            f"{len(cost_variance_df):,}" if len(cost_variance_df) > 0 else 'N/A (no WhsCode)',
            f"{n_clusters}",
            f"{noise_cnt:,}",
        ]
    }
    pd.DataFrame(exec_data).to_excel(writer, sheet_name='Executive Summary', index=False)

    # ── Write date value to E1 ────────────────────────────────────────────────
    ws_exec = writer.sheets['Executive Summary']
    ws_exec['E1'] = 'Last Update/Refresh Time From SAP :'
    ws_exec['F1'] = dt
    ws_exec['F1'].number_format = 'YYYY-MM-DD HH:MM:SS'
    
    all_cols = [
        'ItemCode','ItemName','WhsCode','ItmsGrpCod','ItemGroupName','CardCode','Manufacturer',
        'OnHand','ItemCost','RetailPrice','InventoryValue','Profit','MarginPct',
        'TotalSold_5Y','TotalSalesValue_5Y','MonthlySales',
        'ABC_Class','ABC_Item_Level','XYZ_Class','ABC_XYZ',
        'Velocity_Trend','VelocityRatio','Sales_Slope_5Y',
        'WMA_Forecast_12M','TrendAdj_Forecast_12M','Forecast_Signal',
        'Obsolescence_Risk','ObsolescenceScore',
        'ReorderPoint','SafetyStock','Reorder_Alert','EOQ',
        'Turnover_Rate','Turnover_Rate_Raw','DSI','Turnover_Category',
        'InvestedCapital','LiquidationDisc','CashRelease_If_Sold','WriteOff_Risk_Value','WriteOff_Candidate',
        'NPV_Liquidate_Now','NPV_Liquidate_3M','NPV_Liquidate_6M','NPV_Liquidate_12M',
        'Best_Liquidation_Window','NPV_Cost_of_Waiting_6M',
        'PriceRealizationPct','Margin_Squeeze','DiscountLoss',
        'PriceDrift_Pct','Supplier_Price_Trend',
        'Is_Seasonal','Seasonality_Strength','Peak_Period','SeasonalityIndex',
        'Whs_DemandShare_Pct','Optimal_OnHand_Whs','Inventory_Imbalance','Rebalance_Action',
        'ML_Cluster','ML_Segment','Is_Anomaly','Anomaly_Score',
        'Data_Quality_Flag','Advanced_Action','WebActive','Status',
    ]
    out_cols = [c for c in all_cols if c in df.columns]
    df[out_cols].to_excel(writer, sheet_name='All Products', index=False)

    abc_xyz_matrix = df.groupby('ABC_XYZ').agg(
        Count              = ('ItemCode',          'count'),
        TotalSold_5Y       = ('TotalSold_5Y',      'sum'),
        TotalSalesValue_5Y = ('TotalSalesValue_5Y','sum'),
        AvgMarginPct       = ('MarginPct',         'mean'),
        AvgTurnover        = ('Turnover_Rate',     'mean'),
        InventoryValue= ('InventoryValue',    'sum'),
    ).reset_index().sort_values('TotalSalesValue_5Y', ascending=False)
    abc_xyz_descriptions = {
        'AX': 'Star Product — High sales and stable | Highest inventory priority',
        'AY': 'High Value / Variable — High sales with moderate fluctuations | Higher safety stock',
        'AZ': 'High Value / Unpredictable — High sales but unpredictable | Difficult to plan',
        'BX': 'Stable Mid-Tier — Moderate sales and stable | Standard management',
        'BY': 'Variable Mid-Tier — Moderate sales with fluctuations | Periodic monitoring',
        'BZ': 'Unpredictable Mid-Tier — Moderate sales and unstable | Inventory review needed',
        'CX': 'Low / Stable — Low sales but stable | Keep minimum stock',
        'CY': 'Low / Variable — Low sales with fluctuations | Stock reduction recommended',
        'CZ': 'Dead / Obsolete Risk — Low sales and unstable | Candidate for removal or liquidation',
    }
    abc_xyz_matrix['Description'] = abc_xyz_matrix['ABC_XYZ'].map(abc_xyz_descriptions).fillna('—')
    abc_xyz_matrix.to_excel(writer, sheet_name='ABC-XYZ Matrix', index=False)

    dq_cols = ['ItemCode','ItemName','WhsCode','OnHand','InventoryValue',
               'ItemCost','RetailPrice','MarginPct','Turnover_Rate_Raw','Turnover_Rate','Data_Quality_Flag']
    dq_cols = [c for c in dq_cols if c in df.columns]
    dq_df   = df[df['Data_Quality_Flag'] != 'OK'][dq_cols].copy()
    if len(dq_df) > 0:
        dq_df.to_excel(writer, sheet_name='Data Quality Issues', index=False)

    reorder_cols = ['ItemCode','ItemName','WhsCode','OnHand','MinStock',
                    'ReorderPoint','SafetyStock','EOQ','MonthlySales',
                    'WMA_Forecast_12M','TrendAdj_Forecast_12M','Reorder_Alert']
    reorder_cols = [c for c in reorder_cols if c in df.columns]
    df[df['Reorder_Alert'] == 'REORDER NOW'][reorder_cols]\
        .sort_values('OnHand').to_excel(writer, sheet_name='Reorder Now', index=False)

    obs_cols = ['ItemCode','ItemName','WhsCode','OnHand','InventoryValue',
                'DaysSinceLastSale','TotalSold_5Y','Obsolescence_Risk','ObsolescenceScore',
                'WriteOff_Candidate','WriteOff_Risk_Value','LiquidationDisc',
                'CashRelease_If_Sold','Best_Liquidation_Window','Advanced_Action']
    obs_cols = [c for c in obs_cols if c in df.columns]
    df[df['Obsolescence_Risk'].isin(['Critical','High'])][obs_cols]\
        .sort_values('ObsolescenceScore', ascending=False)\
        .to_excel(writer, sheet_name='Obsolescence & WriteOff', index=False)

    cash_cols = ['ItemCode','ItemName','WhsCode','OnHand','InvestedCapital',
                 'LiquidationDisc','CashRelease_If_Sold',
                 'NPV_Liquidate_Now','NPV_Liquidate_3M','NPV_Liquidate_6M','NPV_Liquidate_12M',
                 'NPV_Cost_of_Waiting_6M','NPV_Cost_of_Waiting_12M',
                 'Best_Liquidation_Window','Obsolescence_Risk']
    cash_cols = [c for c in cash_cols if c in df.columns]
    cash_df   = df[df['Obsolescence_Risk'].isin(['Critical','High'])][cash_cols]\
        .sort_values('NPV_Cost_of_Waiting_6M', ascending=False).copy()
    cash_df.to_excel(writer, sheet_name='Cash Flow Release', index=False)
    ws_cash  = writer.sheets['Cash Flow Release']
    last_row = len(cash_df) + 2
    ws_cash.cell(last_row, 1, 'TOTAL')
    ws_cash.cell(last_row, 5, f"${cash_df['InvestedCapital'].sum():,.0f}" if 'InvestedCapital' in cash_df.columns else '')
    ws_cash.cell(last_row, 7, f"${cash_df['CashRelease_If_Sold'].sum():,.0f}" if 'CashRelease_If_Sold' in cash_df.columns else '')

    if 'WMA_Forecast_12M' in df.columns:
        fc_cols = ['ItemCode','ItemName','WhsCode','ItemGroupName',
                   'QtySold_0_12M','QtySold_12_24M',
                   'MonthlySales','WMA_Forecast_12M','TrendAdj_Forecast_12M',
                   'Sales_Slope_5Y','Forecast_vs_Actual_Pct','Forecast_Signal',
                   'ABC_Class','XYZ_Class']
        fc_cols = [c for c in fc_cols if c in df.columns]
        df[fc_cols].sort_values('TrendAdj_Forecast_12M', ascending=False)\
            .to_excel(writer, sheet_name='Demand Forecast', index=False)

    if 'Rebalance_Action' in df.columns:
        reb_cols = ['ItemCode','ItemName','WhsCode','OnHand',
                    'Whs_DemandShare_Pct','Optimal_OnHand_Whs',
                    'Inventory_Imbalance','Rebalance_Action',
                    'TotalSold_5Y','ABC_Class']
        reb_cols = [c for c in reb_cols if c in df.columns]
        df[df['Rebalance_Action'] != 'Balanced'][reb_cols]\
            .sort_values('Inventory_Imbalance', ascending=False)\
            .to_excel(writer, sheet_name='Whs Rebalancing', index=False)

    if 'Margin_Squeeze' in df.columns:
        margin_cols = ['ItemCode','ItemName','RetailPrice','LastSalePrice','VendorPrice',
                       'ItemCost','Profit','MarginPct','PriceRealizationPct','Margin_Squeeze','DiscountLoss']
        margin_cols = [c for c in margin_cols if c in df.columns]
        df[df['Margin_Squeeze'].str.startswith('Severe', na=False)][margin_cols]\
            .sort_values('DiscountLoss', ascending=False)\
            .to_excel(writer, sheet_name='Margin Squeeze', index=False)

    if 'CardCode' in df.columns and 'sup_agg' in dir():
        sup_agg.sort_values('Supplier_Score', ascending=False)\
            .to_excel(writer, sheet_name='Supplier Performance', index=False)

    if 'Supplier_Price_Trend' in df.columns:
        sup_cols = ['ItemCode','ItemName','CardCode','Manufacturer',
                    'AvgPurchasePrice','LastPurchasePrice','PriceDrift_Pct','Supplier_Price_Trend',
                    'TotalPurchaseValue_5Y','TotalPurchased_5Y']
        sup_cols = [c for c in sup_cols if c in df.columns]
        df[df['Supplier_Price_Trend'].str.contains('Increasing', na=False)][sup_cols]\
            .sort_values('PriceDrift_Pct', ascending=False)\
            .to_excel(writer, sheet_name='Supplier Price Drift', index=False)

    if 'WhsCode' in df.columns:
        whs_summary = df.groupby('WhsCode').agg(
            SKUs_Count           = ('ItemCode',          'count'),
            InventoryValue  = ('InventoryValue',    'sum'),
            TotalSold_5Y         = ('TotalSold_5Y',      'sum'),
            TotalSalesValue_5Y   = ('TotalSalesValue_5Y','sum'),
            AvgTurnover          = ('Turnover_Rate',     'mean'),
            AvgObsolescenceScore = ('ObsolescenceScore', 'mean'),
            Critical_SKUs        = ('Obsolescence_Risk', lambda x: (x == 'Critical').sum()),
            Reorder_SKUs         = ('Reorder_Alert',     lambda x: (x == 'REORDER NOW').sum()),
            Transfer_Out_Count   = ('Rebalance_Action',  lambda x: (x == 'Transfer Out').sum()),
            Transfer_In_Count    = ('Rebalance_Action',  lambda x: (x == 'Transfer In').sum()),
        ).reset_index().sort_values('InventoryValue', ascending=False)
        whs_summary.to_excel(writer, sheet_name='Warehouse Comparison', index=False)

    if 'ItemGroupName' in df.columns:
        cat_summary = df.groupby('ItemGroupName').agg(
            SKUs_Count          = ('ItemCode',               'count'),
            InventoryValue = ('InventoryValue',         'sum'),
            TotalSold_5Y        = ('TotalSold_5Y',           'sum'),
            TotalSalesValue_5Y  = ('TotalSalesValue_5Y',     'sum'),
            AvgMarginPct        = ('MarginPct',              'mean'),
            AvgTurnover         = ('Turnover_Rate',          'mean'),
            WriteOff_Value      = ('WriteOff_Risk_Value',    'sum'),
            Seasonal_Count      = ('Is_Seasonal',            'sum'),
            Forecast_Total_12M  = ('TrendAdj_Forecast_12M', 'sum') if 'TrendAdj_Forecast_12M' in df.columns else ('ItemCode','count'),
        ).reset_index().sort_values('TotalSalesValue_5Y', ascending=False)
        cat_summary.to_excel(writer, sheet_name='Category Analysis', index=False)

    if 'WebActive' in df.columns:
        web_summary = df.groupby('WebActive').agg(
            SKUs_Count           = ('ItemCode',          'count'),
            TotalSold_5Y         = ('TotalSold_5Y',      'sum'),
            TotalSalesValue_5Y   = ('TotalSalesValue_5Y','sum'),
            AvgMarginPct         = ('MarginPct',         'mean'),
            AvgTurnover          = ('Turnover_Rate',     'mean'),
            AvgObsolescenceScore = ('ObsolescenceScore', 'mean'),
        ).reset_index()
        web_summary['WebActive'] = web_summary['WebActive'].map({1: 'Active', 0: 'Not Active'})
        web_summary.to_excel(writer, sheet_name='WebActive Impact', index=False)

    if 'Status' in df.columns:
        inactive_cols = ['ItemCode','ItemName','WhsCode','OnHand','IsCommited','OnOrder','InventoryValue',
                         'TotalSold_5Y','DaysSinceLastSale','Obsolescence_Risk',
                         'Best_Liquidation_Window','Advanced_Action']
        inactive_cols = [c for c in inactive_cols if c in df.columns]
        df[(df['Status'] == 'Inactive') & (
            (df['OnHand'] != 0) | (df['IsCommited'] != 0) | (df['OnOrder'] != 0)
        )][inactive_cols].sort_values('OnHand', ascending=False)\
            .to_excel(writer, sheet_name='Inactive Exceptions', index=False)

    # ── Cost Variance Sheets ──────────────────────────────────────────────────
    if len(cost_variance_df) > 0:
        cv_summary_cols = [
            'ItemCode','ItemName','Whs_Count',
            'Cost_Min','Cost_Max','Cost_Mean','Cost_Std',
            'Cost_Variance_Pct','Cost_Gap_Abs',
            'TotalOnHand','TotalInvValue','Financial_Impact','Variance_Severity'
        ]
        cost_variance_df[cv_summary_cols].to_excel(
            writer, sheet_name='Cost Variance Across Whs', index=False
        )
        if len(cost_detail_df) > 0:
            cost_detail_df.to_excel(
                writer, sheet_name='Cost Variance — Detail', index=False
            )

    # ── ML Sheets — HDBSCAN ───────────────────────────────────────────────────
    hdbscan_info = pd.DataFrame({
        'Metric': [
            'Algorithm', 'Total Clusters Found', 'Noise Points (unclustered)',
            'min_cluster_size', 'min_samples',
        ],
        'Value': [
            'HDBSCAN', n_clusters, noise_cnt, min_cls, 10,
        ]
    })
    hdbscan_info.to_excel(writer, sheet_name='ML Clusters_HDBSCAN', index=False, startrow=0)

    importance_df = importance.reset_index()
    importance_df.columns = ['Feature', 'Importance']
    importance_df.to_excel(writer, sheet_name='ML Clusters_HDBSCAN', index=False,
                            startrow=len(hdbscan_info) + 3)

    cp_out_cols = ['ML_Cluster', 'ML_Label'] + features
    cp_out_cols = [c for c in cp_out_cols if c in cluster_profile.columns]
    cluster_profile[cp_out_cols].to_excel(
        writer, sheet_name='ML Clusters_HDBSCAN', index=False,
        startrow=len(hdbscan_info) + len(importance_df) + 6
    )

    ml_summary_out = df.groupby(['ML_Cluster','ML_Segment']).agg(
        Count                = ('ItemCode',          'count'),
        TotalSold_5Y         = ('TotalSold_5Y',      'sum'),
        AvgMarginPct         = ('MarginPct',         'mean'),
        AvgTurnover          = ('Turnover_Rate',     'mean'),
        InventoryValue  = ('InventoryValue',    'sum'),
        AvgObsolescenceScore = ('ObsolescenceScore', 'mean'),
        AvgVelocityRatio     = ('VelocityRatio',     'mean'),
    ).reset_index().sort_values('ML_Cluster')
    ml_summary_out.to_excel(
        writer, sheet_name='ML Clusters_HDBSCAN', index=False,
        startrow=len(hdbscan_info) + len(importance_df) + len(cluster_profile) + 9
    )

    ml_detail_cols = ['ItemCode','ItemName','WhsCode','ML_Cluster','ML_Segment',
                       'TotalSold_5Y','MarginPct','Turnover_Rate','VelocityRatio',
                       'ObsolescenceScore','Obsolescence_Risk']
    ml_detail_cols = [c for c in ml_detail_cols if c in df.columns]
    df[ml_detail_cols].sort_values(['ML_Cluster','TotalSold_5Y'], ascending=[True,False])\
        .to_excel(writer, sheet_name='ML Cluster Detail', index=False)

    anomaly_out_cols = ['ItemCode','ItemName','WhsCode',
                         'TotalSold_5Y','VelocityRatio','MarginPct','Turnover_Rate',
                         'Anomaly_Score','Obsolescence_Risk','Advanced_Action']
    anomaly_out_cols = [c for c in anomaly_out_cols if c in df.columns]
    df[df['Is_Anomaly']][anomaly_out_cols].sort_values('Anomaly_Score')\
        .to_excel(writer, sheet_name='Anomalies', index=False)

    seasonal_cols = ['ItemCode','ItemName','ItemGroupName','Is_Seasonal',
                      'Seasonality_Strength','SeasonalityIndex','Peak_Period',
                      'TotalSold_5Y','MonthlySales'] + sold_cols
    seasonal_cols = [c for c in seasonal_cols if c in df.columns]
    df[df['Is_Seasonal']][seasonal_cols].sort_values('SeasonalityIndex', ascending=False)\
        .to_excel(writer, sheet_name='Seasonal Items', index=False)

    vel_cols = ['ItemCode','ItemName','WhsCode','Velocity_Trend','VelocityRatio',
                 'Sales_Slope_5Y','ABC_Class'] + sold_cols
    vel_cols = [c for c in vel_cols if c in df.columns]
    df[vel_cols].sort_values('Sales_Slope_5Y', ascending=False)\
        .to_excel(writer, sheet_name='Velocity Trends', index=False)

    sheet_colors = {
        'Executive Summary':        'FF5733',
        'All Products':             '33FF57',
        'ABC-XYZ Matrix':           '3357FF',
        'Data Quality Issues':      'CC0000',
        'Reorder Now':              'FF33A6',
        'Obsolescence & WriteOff':  'A633FF',
        'Cash Flow Release':        '33FFF0',
        'Demand Forecast':          '4CAF50',
        'Whs Rebalancing':          'FF9800',
        'Margin Squeeze':           'FFC133',
        'Supplier Performance':     '00BCD4',
        'Supplier Price Drift':     'FF3333',
        'Warehouse Comparison':     '33FF8A',
        'Category Analysis':        'FF8A33',
        'WebActive Impact':         '338AFF',
        'Inactive Exceptions':      'B4AEE0',
        'Cost Variance Across Whs': 'E74C3C',
        'Cost Variance — Detail':   'C0392B',
        'ML Clusters_HDBSCAN':      '8A33FF',
        'ML Cluster Detail':        '33FFB5',
        'Anomalies':                'FF3380',
        'Seasonal Items':           'FFD433',
        'Velocity Trends':          '33D4FF',
    }
    for sname, color in sheet_colors.items():
        if sname not in writer.sheets:
            continue
        ws = writer.sheets[sname]
        fmt_header(ws, color)
        for cell in ws[1]:
            cell.font = Font(color="000000", bold=True)
        if sname == 'ABC-XYZ Matrix':
            auto_width(ws, col_overrides={'Description': 80})
        else:
            auto_width(ws)
        ws.freeze_panes = 'B2'
        ws.sheet_properties.tabColor = f'FF{color}'

# ─────────────────────────────────────────
# 20. PRINT SUMMARY
# ─────────────────────────────────────────
print(f"\n Analysis completed successfully")
print(f" Output: {OUTPUT_FILE}")
print(f"\n Executive Summary:")
print(f"   • Total Inventory Value      : ${df['InventoryValue'].sum():,.0f}")
print(f"   • Potential Cash Recovery    : ${df[df['Obsolescence_Risk'].isin(['Critical','High'])]['CashRelease_If_Sold'].sum():,.0f}")
print(f"   • NPV Cost of Waiting 6M     : ${df['NPV_Cost_of_Waiting_6M'].sum():,.0f}")
print(f"   • Total SKUs/Whs Records     : {len(df):,}")
print(f"   • High Obsolescence Risk     : {df['Obsolescence_Risk'].isin(['Critical','High']).sum():,}")
print(f"   • Urgent Reorders            : {(df['Reorder_Alert'] == 'REORDER NOW').sum():,}")
print(f"   • Anomalies Detected         : {df['Is_Anomaly'].sum():,}")
print(f"   • Seasonal Items             : {df['Is_Seasonal'].sum():,}")
print(f"   • Data Quality Issues        : {(df['Data_Quality_Flag'] != 'OK').sum():,}")
print(f"   • HDBSCAN Clusters Found     : {n_clusters}")
print(f"   • HDBSCAN Noise Points       : {noise_cnt:,}")
print(f"   • Items w/ Cost Variance ≥5% : {len(cost_variance_df):,}" if len(cost_variance_df) > 0 else "   • Cost Variance              : N/A")
if 'Rebalance_Action' in df.columns:
    print(f"   • Items Needing Rebalancing  : {(df['Rebalance_Action'] != 'Balanced').sum():,}")
print(f"   • Sheets Generated           : {len(sheet_colors):,}")
